In [1]:
%pip install mlxtend

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 6.4 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

from mlxtend.frequent_patterns import (
    apriori,
    fpgrowth,
    association_rules
)

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

data = pd.read_csv(
    output_dir / "sparrow_model_data.csv",
    dtype={"ID": "string"}
)

mining_data = data.loc[
    data["Split"] == "train"
].copy()

print("Mining records:", len(mining_data))

Mining records: 501


In [ ]:
# change numeric data to categories
# buidling-level record 1 = transaction 1
association_features = [
    "BuildingHeight",
    "BuildingWidth",
    "Altitude",
    "AveNoise",
    "temp",
    "prec",
    "win"
]

binned = pd.DataFrame(index=mining_data.index)
bin_records = []

for column in association_features:
    q1, q2 = mining_data[column].quantile(
        [1 / 3, 2 / 3]
    )

    if q1 >= q2:
        raise ValueError(
            f"{column}: cannot create three distinct bins"
        )

    binned[column] = pd.cut(
        mining_data[column],
        bins=[-np.inf, q1, q2, np.inf],
        labels=["Low", "Medium", "High"],
        include_lowest=True
    )

    bin_records.append({
        "Feature": column,
        "LowUpperInclusive": q1,
        "MediumUpperInclusive": q2
    })

binned["NestCountGroup"] = (
    mining_data["HigherNestCount"]
    .map({0: "Lower", 1: "Higher"})
)

bin_boundaries = pd.DataFrame(bin_records)

display(bin_boundaries)
display(binned.head())

# low: value <= traning 33.3% quantile
# medium: 33.3 <= value <= 66.7
# high: value > 66.7

,Feature,LowUpperInclusive,MediumUpperInclusive
0,BuildingHeight,18.000000,24.00
1,BuildingWidth,40.000000,60.00
2,Altitude,784.000000,1179.21
3,AveNoise,49.000000,54.00
4,temp,16.193333,17.08
5,prec,43.000000,59.67
6,win,2.430000,2.53


,BuildingHeight,BuildingWidth,Altitude,AveNoise,temp,prec,win,NestCountGroup
0,High,Medium,Low,Low,High,High,Medium,Higher
1,High,High,Low,Low,High,High,Medium,Higher
2,High,Medium,Low,Low,High,High,Medium,Higher
3,High,Medium,Low,Low,High,High,Medium,Higher
4,High,Medium,Low,Low,High,High,Medium,Higher


In [4]:
# boolean transaction table
basket = pd.get_dummies(
    binned,
    prefix_sep="=",
    dtype=bool
)

display(basket.head())

print("Basket shape:", basket.shape)

assert not binned.isna().any().any()
assert basket.sum(axis=1).eq(8).all()

,BuildingHeight=Low,BuildingHeight=Medium,BuildingHeight=High,BuildingWidth=Low,BuildingWidth=Medium,BuildingWidth=High,Altitude=Low,Altitude=Medium,Altitude=High,AveNoise=Low,...,temp=Medium,temp=High,prec=Low,prec=Medium,prec=High,win=Low,win=Medium,win=High,NestCountGroup=Higher,NestCountGroup=Lower
0,False,False,True,False,True,False,True,False,False,True,...,False,True,False,False,True,False,True,False,True,False
1,False,False,True,False,False,True,True,False,False,True,...,False,True,False,False,True,False,True,False,True,False
2,False,False,True,False,True,False,True,False,False,True,...,False,True,False,False,True,False,True,False,True,False
3,False,False,True,False,True,False,True,False,False,True,...,False,True,False,False,True,False,True,False,True,False
4,False,False,True,False,True,False,True,False,False,True,...,False,True,False,False,True,False,True,False,True,False


Basket shape: (501, 23)


In [5]:
# run apriori and fp-growth
MIN_SUPPORT = 0.10
MAX_ITEMSET_LENGTH = 3

start = perf_counter()

apriori_itemsets = apriori(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH
)

apriori_seconds = perf_counter() - start

start = perf_counter()

fp_itemsets = fpgrowth(
    basket,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_ITEMSET_LENGTH
)

fp_seconds = perf_counter() - start

algorithm_comparison = pd.DataFrame([
    {
        "Algorithm": "Apriori",
        "FrequentItemsets": len(apriori_itemsets),
        "Seconds": apriori_seconds
    },
    {
        "Algorithm": "FP-growth",
        "FrequentItemsets": len(fp_itemsets),
        "Seconds": fp_seconds
    }
])

display(algorithm_comparison)

,Algorithm,FrequentItemsets,Seconds
0,Apriori,217,0.006676
1,FP-growth,217,0.010431


In [6]:
# check the rresults of 2 algorithms
a = (
    apriori_itemsets
    .set_index("itemsets")["support"]
    .to_dict()
)

b = (
    fp_itemsets
    .set_index("itemsets")["support"]
    .to_dict()
)

assert a.keys() == b.keys()

assert all(
    np.isclose(a[itemset], b[itemset])
    for itemset in a
)

print("Verified: itemsets and supports match.")

Verified: itemsets and supports match.


In [7]:
# generate association rules
MIN_CONFIDENCE = 0.60

rules = association_rules(
    fp_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

# Nest-count group ကို conclusion အဖြစ်ထားတဲ့ rules
target_items = {
    "NestCountGroup=Lower",
    "NestCountGroup=Higher"
}

nest_rules = rules.loc[
    rules["consequents"].apply(
        lambda items:
            len(items) == 1
            and items.issubset(target_items)
    )
    &
    rules["antecedents"].apply(
        lambda items:
            items.isdisjoint(target_items)
    )
].copy()

# Positive association ရှိတဲ့ rules ကို shortlist လုပ်မယ်
positive_rules = nest_rules.loc[
    nest_rules["lift"] > 1
].sort_values(
    ["lift", "confidence", "support"],
    ascending=False
)

display(
    positive_rules[[
        "antecedents",
        "consequents",
        "support",
        "confidence",
        "lift"
    ]].head(10)
)

,antecedents,consequents,support,confidence,lift
66,"frozenset({Altitude=High, BuildingWidth=Low})",frozenset({NestCountGroup=Lower}),0.103792,0.912281,1.655988
37,"frozenset({Altitude=Low, prec=High})",frozenset({NestCountGroup=Higher}),0.155689,0.742857,1.654095
5,"frozenset({BuildingHeight=High, Altitude=Low})",frozenset({NestCountGroup=Higher}),0.131737,0.741573,1.651236
17,"frozenset({BuildingHeight=High, temp=High})",frozenset({NestCountGroup=Higher}),0.113772,0.721519,1.606582
62,"frozenset({Altitude=High, BuildingHeight=Low})",frozenset({NestCountGroup=Lower}),0.135729,0.883117,1.603049
13,"frozenset({Altitude=Low, temp=High})",frozenset({NestCountGroup=Higher}),0.199601,0.719424,1.601918
93,"frozenset({temp=Low, win=Low})",frozenset({NestCountGroup=Lower}),0.109780,0.873016,1.584714
89,"frozenset({BuildingWidth=Medium, temp=Low})",frozenset({NestCountGroup=Lower}),0.119760,0.869565,1.578450
74,"frozenset({Altitude=High, temp=Low})",frozenset({NestCountGroup=Lower}),0.201597,0.863248,1.566983
65,"frozenset({Altitude=High, AveNoise=Low})",frozenset({NestCountGroup=Lower}),0.123752,0.861111,1.563104


In [8]:
# Checking rule with real math formula
if positive_rules.empty:
    print("No positive nest rules meet the current thresholds.")

else:
    selected = positive_rules.iloc[0]

    antecedent = sorted(selected["antecedents"])
    consequent = sorted(selected["consequents"])

    has_a = basket[antecedent].all(axis=1)
    has_b = basket[consequent].all(axis=1)

    n = len(basket)
    count_a = int(has_a.sum())
    count_b = int(has_b.sum())
    count_ab = int((has_a & has_b).sum())

    support = count_ab / n
    confidence = count_ab / count_a
    lift = confidence / (count_b / n)

    print("IF:", antecedent)
    print("THEN:", consequent)

    print("Total records:", n)
    print("A records:", count_a)
    print("B records:", count_b)
    print("A and B records:", count_ab)

    print("Support:", round(support, 4))
    print("Confidence:", round(confidence, 4))
    print("Lift:", round(lift, 4))

    assert np.isclose(support, selected["support"])
    assert np.isclose(confidence, selected["confidence"])
    assert np.isclose(lift, selected["lift"])

IF: ['Altitude=High', 'BuildingWidth=Low']
THEN: ['NestCountGroup=Lower']
Total records: 501
A records: 57
B records: 276
A and B records: 52
Support: 0.1038
Confidence: 0.9123
Lift: 1.656


In [9]:
# save the result
def export_rules(frame, filename):
    exported = frame.copy()

    for column in ["antecedents", "consequents"]:
        exported[column] = exported[column].apply(
            lambda items: " AND ".join(sorted(items))
        )

    exported.to_csv(
        output_dir / filename,
        index=False
    )

export_rules(
    nest_rules,
    "nest_association_rules.csv"
)

export_rules(
    positive_rules,
    "positive_nest_rules.csv"
)

bin_boundaries.to_csv(
    output_dir / "association_bin_boundaries.csv",
    index=False
)

algorithm_comparison.to_csv(
    output_dir / "association_algorithm_comparison.csv",
    index=False
)

print("Association results saved.")

Association results saved.
